<a href="https://colab.research.google.com/github/kilianodonell-cmd/Q3_Durban/blob/main/Field_Map_Deep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# Mount Google Drive (only in Colab)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

print("Setup complete.")

Mounted at /content/drive
Setup complete.


In [2]:
import os, json, numpy as np
import geopandas as gpd
import rasterio
from rasterio.warp import reproject, Resampling, calculate_default_transform
import folium
from folium import LayerControl
from PIL import Image
import base64, io
import matplotlib.pyplot as plt
import pandas as pd


In [5]:
# ============================================================
# CELL 1 — Setup
# Field Map — Housing Suitability
# Mzinyati Stream Catchment, eThekwini Municipality
# ============================================================

# Load config (created by MCA notebook CELL 2)
CONFIG_PATH = '/content/drive/MyDrive/Durban/outputs/field_map_config.json'

with open(CONFIG_PATH) as f:
    config = json.load(f)

OUTPUT_ROOT    = config['OUTPUT_ROOT']
TARGET_CRS     = config['TARGET_CRS']
SCENARIOS      = config['SCENARIOS']
SUIT_COLORS    = config['SUIT_COLORS']
SUIT_LABELS    = config['SUIT_LABELS']
AOI_FIELD      = config['AOI_FIELD']
AOI_VALUE      = config['AOI_VALUE']
CATCHMENTS_PATH = config['CATCHMENTS_PATH']

RASTERS_DIR    = os.path.join(OUTPUT_ROOT, 'rasters')
BUILDINGS_PATH = os.path.join(OUTPUT_ROOT, 'outputs', 'at_risk_buildings.gpkg')
CLIPPED_DIR    = os.path.join(OUTPUT_ROOT, 'clipped')

print("=" * 60)
print("FIELD MAP SETUP")
print("=" * 60)
print(f"  Output root: {OUTPUT_ROOT}")
print(f"  Scenarios:   {list(SCENARIOS.keys())}")

# Check that required files exist
print("\n  Checking files...")
missing = []

for scenario in SCENARIOS:
    raster_path = os.path.join(RASTERS_DIR, f"suitability_{scenario}.tif")
    if os.path.exists(raster_path):
        print(f"    ✓ suitability_{scenario}.tif")
    else:
        print(f"    ✗ suitability_{scenario}.tif MISSING")
        missing.append(raster_path)

constraint_path = os.path.join(RASTERS_DIR, 'constraint_mask.tif')
if os.path.exists(constraint_path):
    print(f"    ✓ constraint_mask.tif")
else:
    print(f"    ✗ constraint_mask.tif MISSING")
    missing.append(constraint_path)

if os.path.exists(BUILDINGS_PATH):
    print(f"    ✓ at_risk_buildings.gpkg")
else:
    print(f"    ✗ at_risk_buildings.gpkg MISSING")
    missing.append(BUILDINGS_PATH)

if missing:
    print(f"\n  ⚠️ WARNING: {len(missing)} files missing. Run MCA notebook first.")
else:
    print(f"\n  ✓ All files ready. Proceed to CELL 2.")

print("=" * 60)

FIELD MAP SETUP
  Output root: /content/drive/MyDrive/Durban/outputs
  Scenarios:   ['hazard_focused', 'balanced', 'infrastructure_focused']

  Checking files...
    ✓ suitability_hazard_focused.tif
    ✓ suitability_balanced.tif
    ✓ suitability_infrastructure_focused.tif
    ✓ constraint_mask.tif
    ✓ at_risk_buildings.gpkg

  ✓ All files ready. Proceed to CELL 2.


In [20]:
# ============================================================
# CELL 16 — FINAL CORRECT MAP
# ============================================================

import os
import numpy as np
import folium
from folium import plugins
import geopandas as gpd
import rasterio
from rasterio.features import shapes
from shapely.geometry import shape
from IPython.display import IFrame, display

print("=" * 60)
print("FINAL MAP")
print("=" * 60)

# ------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------
AT_RISK_PATH = os.path.join(OUTPUT_ROOT, "outputs", "at_risk_buildings.gpkg")
CONSTRAINT_RASTER = os.path.join(RASTERS_DIR, "constraint_mask.tif")

# Load buildings
buildings = gpd.read_file(AT_RISK_PATH).to_crs(4326)
buildings["geometry"] = buildings.geometry.simplify(0.00001, preserve_topology=True)

# Split building groups
constraint_buildings = buildings[buildings["in_constraint"] == True].copy()
non_constraint_buildings = buildings[buildings["in_constraint"] != True].copy()

# Load AOI
catchments = gpd.read_file(CATCHMENTS_PATH)
aoi = catchments[catchments[AOI_FIELD] == AOI_VALUE].copy().to_crs(4326)
aoi["geometry"] = aoi.geometry.simplify(0.00001, preserve_topology=True)

# ------------------------------------------------------------
# LOAD CONSTRAINT MASK FROM RASTER
# NOTE: In Cell 11 -> 0 = constraint, 1 = buildable, 255 = nodata
# ------------------------------------------------------------
constraint_mask_gdf = gpd.GeoDataFrame()

with rasterio.open(CONSTRAINT_RASTER) as src:
    mask_data = src.read(1)
    transform = src.transform
    crs = src.crs
    nodata = src.nodata

    print("Unique values in constraint raster:", np.unique(mask_data))

    if nodata is not None:
        valid_mask = mask_data != nodata
    else:
        valid_mask = np.ones(mask_data.shape, dtype=bool)

    results = []
    for geom, value in shapes(mask_data, mask=valid_mask, transform=transform):
        if value == 0:   # 0 = excluded / constraint area
            results.append(shape(geom))

    if results:
        constraint_mask_gdf = gpd.GeoDataFrame(
            geometry=results,
            crs=crs
        ).to_crs(4326)
        constraint_mask_gdf["geometry"] = constraint_mask_gdf.geometry.simplify(
            0.00001, preserve_topology=True
        )

# ------------------------------------------------------------
# MAP CENTER
# ------------------------------------------------------------
center = aoi.union_all().centroid
bounds = aoi.total_bounds

print(f"AOI: {AOI_VALUE}")
print(f"Total buildings: {len(buildings):,}")
print(f"Constraint buildings: {len(constraint_buildings):,}")
print(f"Non-constraint buildings: {len(non_constraint_buildings):,}")
print(f"Constraint polygons: {len(constraint_mask_gdf):,}")

# ------------------------------------------------------------
# CREATE MAP
# ------------------------------------------------------------
m = folium.Map(
    location=[center.y, center.x],
    zoom_start=14,
    tiles="https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png",
    attr='&copy; <a href="https://www.openstreetmap.org/copyright">OSM</a> &copy; CARTO',
    control_scale=True
)

# ------------------------------------------------------------
# LAYER 1: AOI BOUNDARY
# ------------------------------------------------------------
folium.GeoJson(
    aoi,
    name="AOI boundary",
    style_function=lambda feat: {
        "fillColor": "none",
        "color": "black",
        "weight": 2,
        "dashArray": "5, 5"
    }
).add_to(m)

# ------------------------------------------------------------
# LAYER 2: CONSTRAINT MASK
# ------------------------------------------------------------
if len(constraint_mask_gdf) > 0:
    folium.GeoJson(
        constraint_mask_gdf,
        name="Constraint mask (area)",
        style_function=lambda feat: {
            "fillColor": "#dc2626",
            "color": "#991b1b",
            "weight": 0.8,
            "fillOpacity": 0.35
        },
        tooltip="CONSTRAINT ZONE"
    ).add_to(m)

# ------------------------------------------------------------
# LAYER 3: CONSTRAINT BUILDINGS
# ------------------------------------------------------------
if len(constraint_buildings) > 0:
    folium.GeoJson(
        constraint_buildings,
        name="Constraint buildings",
        style_function=lambda feat: {
            "fillColor": "#7f1a1a",
            "color": "#450a0a",
            "weight": 0.3,
            "fillOpacity": 0.9
        },
        popup=folium.GeoJsonPopup(
            fields=["display_score_hazard_focused", "raw_score_hazard_focused"],
            aliases=["Display score:", "Raw score:"]
        )
    ).add_to(m)

# ------------------------------------------------------------
# LAYER 4: NON-CONSTRAINT BUILDINGS
# ------------------------------------------------------------
def style_function(feature):
    score = feature["properties"].get("display_score_hazard_focused", -1)
    if score in [1, 2, 3, 4, 5]:
        return {
            "fillColor": SUIT_COLORS[score - 1],
            "color": "#333333",
            "weight": 0.3,
            "fillOpacity": 0.85
        }
    return {
        "fillColor": "#9e9e9e",
        "color": "#333333",
        "weight": 0.3,
        "fillOpacity": 0.85
    }

folium.GeoJson(
    non_constraint_buildings,
    name="Non-constraint buildings",
    style_function=style_function,
    show=True,
    popup=folium.GeoJsonPopup(
        fields=["display_score_hazard_focused", "raw_score_hazard_focused"],
        aliases=["Display score:", "Raw score:"]
    )
).add_to(m)

# ------------------------------------------------------------
# PLUGINS
# ------------------------------------------------------------
plugins.Fullscreen().add_to(m)

# ------------------------------------------------------------
# LEGEND
# ------------------------------------------------------------
legend_html = f"""
<div style="
position: fixed;
bottom: 20px;
left: 20px;
z-index: 9999;
background-color: white;
padding: 10px 12px;
border: 1px solid #ccc;
border-radius: 5px;
font-size: 12px;
font-family: Arial, sans-serif;
">
<b>Suitability (non-constraint)</b><br>
<span style="background:{SUIT_COLORS[0]};">&nbsp;&nbsp;&nbsp;</span> {SUIT_LABELS[0]}<br>
<span style="background:{SUIT_COLORS[1]};">&nbsp;&nbsp;&nbsp;</span> {SUIT_LABELS[1]}<br>
<span style="background:{SUIT_COLORS[2]};">&nbsp;&nbsp;&nbsp;</span> {SUIT_LABELS[2]}<br>
<span style="background:{SUIT_COLORS[3]};">&nbsp;&nbsp;&nbsp;</span> {SUIT_LABELS[3]}<br>
<span style="background:{SUIT_COLORS[4]};">&nbsp;&nbsp;&nbsp;</span> {SUIT_LABELS[4]}<br>
<hr>
<span style="background:#dc2626;">&nbsp;&nbsp;&nbsp;</span> Constraint mask<br>
<span style="background:#7f1a1a;">&nbsp;&nbsp;&nbsp;</span> Constraint buildings
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

# ------------------------------------------------------------
# LAYER CONTROL & FIT
# ------------------------------------------------------------
folium.LayerControl(collapsed=False).add_to(m)
m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------
output_map = os.path.join(OUTPUT_ROOT, "outputs", "final_map.html")
m.save(output_map)

print(f"\n✅ Map saved: {output_map}")
display(IFrame(output_map, width="100%", height="650"))

FINAL MAP
Unique values in constraint raster: [0 1]
AOI: Mzinyati Stream
Total buildings: 22,196
Constraint buildings: 4,268
Non-constraint buildings: 17,928
Constraint polygons: 129

✅ Map saved: /content/drive/MyDrive/Durban/outputs/outputs/final_map.html
